# Forge Market EDA and Study Panel Selection

## 1. Purpose and scope

**Primary Analysis Question**: Can I detect market "anomalies" (whatever that might mean) in [EVE Online](https://www.eveonline.com/)'s trading marketplace?

I am selecting a fixed panel of item markets to study changes in price and trading activity in The Forge, the region containing EVE Online's most popular trading hub, Jita. Following the same markets over time should make it easier to study their behavior without constantly changing which items we're looking at.

I've decided that the *selection window* will be **June 1 through September 30, 2022**, inclusive, because this is a decently long period of time between any significant updates.

The proposed panel combines seven minerals with the top 50 manufactured Tech I products by market-history order count and the top 50 by estimated traded ISK. These items also have manufacturing relationships that we can use to investigate movements across related markets, rather than only looking at each item individually.

This notebook covers selecting these markets, checking their historical data availability, and getting a sense of their characteristics. We'll export the final panel for downstream analysis after documenting any exclusions.

I am also assuming that the latest retained local reference archive is not significantly different from the reference data throughout these few years apart from additions and published-status changes, since that would complicate the analysis significantly.

**Prerequisites for running this notebook**:
- Data ingested from Everef, covering at least the aforementioned selection window along with the reference data
- Running `mise run transform:build` to materialize the DuckDB database locally 

## 2. Data and reference preparation

### 2.1 Analytical setup

Market history and selection item references are accessed through the read-only DuckDB database.

The market history covers The Forge as a whole, not just transactions at Jita. It contains daily summaries of completed trading, rather than the standing buy and sell orders that we'd find in an order book.

In [ ]:
import math
import sys
from contextlib import contextmanager
from datetime import date
from pathlib import Path

import plotly.io as pio
import polars as pl

# Allow imports when the kernel starts at the repository root or either notebook directory.
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "transformation" / "dbt_project.yml").is_file()
)
experiments_root = repo_root / "experiments"
if str(experiments_root) not in sys.path:
    sys.path.insert(0, str(experiments_root))

# all plotting and reference data extraction code are stored away in python modules
from market_analysis.reference_data import (
    find_repo_root, retained_reference_archive, load_reference_records,
    manufacturing_products, manufacturing_links,
)
from market_analysis.plotting.eda import (
    CATEGORY_COLORS, plot_candidate_relationships, plot_activity_distribution,
    plot_activity_ranking, plot_ranked_selection_overlap,
    plot_monthly_history_availability, plot_retained_distributions,
    plot_retained_composition, plot_selection_price_changes,
)
from market_analysis.plotting.production_graph import plot_production_graph
from transform.notebook_access import open_analysis_connection

# Use self-contained HTML output when JupyterLab does not render Plotly MIME output.
pio.renderers.default = "notebook"

REGION_ID = 10000002
SELECTION_START = date(2022, 6, 1)
SELECTION_END = date(2022, 9, 30)
TOP_N = 50
SELECTION_DAYS = (SELECTION_END - SELECTION_START).days + 1

# Set SEED_OUTPUT_PATH and PRODUCTION_LINKS_OUTPUT_PATH to redirect validation exports.

### 2.2 Reference-data preparation

The market history tells us about prices and trading activity, but it doesn't tell us much about the items themselves or how they're manufactured.

For that, we're using the latest retained local archive of [Everef's reference extract](https://docs.everef.net/datasets/reference-data), obtained during the Airflow ingestion process. This gives us item names, Tech classifications, and manufacturing blueprints. We'll use these to identify our candidate products and, later on, connect them to their production inputs and outputs.

In [ ]:
reference_archive_path, reference_acquired_at = retained_reference_archive()
blueprints, reference_types, meta_groups = load_reference_records(reference_archive_path)

## 3. Candidate markets and selection-period activity

Before choosing the study panel, we'll need to identify the eligible manufactured products and compare their activity during the selection window. This should give us a sense of what we're selecting for and how.

### 3.1 Tech I candidates

We'll limit our activity analysis to items with an explicit [Tech I](https://wiki.eveuniversity.org/Tech_and_meta_levels) classification (`meta_group_id = 1` within the reference data) that also appear as manufacturing blueprint products. Tech I items are, broadly speaking, foundational goods in EVE Online:
- They offer basic, everyday ships, equipment, and consumables, used by both new and experienced players
- Some items are sold on the market by NPCs or offered as mission rewards, providing alternative supply sources alongside player manufacturing
- They have some of the simplest production chains in the game, mostly requiring the 7 primary minerals
- They require less [skill](https://wiki.eveuniversity.org/Skills_and_learning) investment to use and produce than their Tech II counterparts
- Many of them are also used as production inputs for more advanced Tech II goods

These characteristics make Tech I products a very useful starting point for studying economic relationships in EVE Online's marketplace. Their relatively simple production chains also give us common inputs through which we can compare otherwise different item markets.

The rankings will only consider these manufactured products. We'll add the seven minerals separately later on so that these common production inputs are represented as well.

Now, in order to make the analysis a bit more manageable, we're going to select the most active/economically interesting items and declare those as our **study panel**.

In [ ]:
product_ids, blueprints_without_products = manufacturing_products(blueprints)

manufacturing_product_ids = pl.DataFrame(
    {"type_id": sorted(product_ids)}, schema={"type_id": pl.Int64}
)
manufacturing_reference_input = pl.DataFrame(
    {
        "Local archive path": [str(reference_archive_path)],
        "Acquired at (UTC)": [reference_acquired_at.isoformat()],
        "Blueprint records": [len(blueprints)],
        "Unique manufacturing products": [manufacturing_product_ids.height],
        "Blueprints without manufacturing products": [len(blueprints_without_products)],
    }
)
manufacturing_reference_input

In [ ]:
@contextmanager
def open_study_connection():
    with open_analysis_connection() as con:
        con.register("manufacturing_product_ids", manufacturing_product_ids)
        yield con

### 3.2 Candidate activity relationships

In order to determine our most economically active/interesting items, we're going to aggregate the activity for each Tech I candidate during the aforementioned *selection window*, then select the most active ones. But before we do that, a quick overview of some of the base features in our data:

- Each row in the market history data represents a given market day, for a given *region*, and for a given *item* (AKA "type")
    - The grain is (`market_date`, `region_id`, `type_id`); note that we only care about The Forge region, already selected via the `dbt` pipeline
- Some of the notable features are:
    - `average`: the volume-weighted average price (VWAP) in ISK per unit
    - `highest`, `lowest`: the highest and lowest transaction prices in ISK per unit
    - `order_count`: the historical order count for that day, not the number of standing buy or sell orders
    - `volume`: the total number of item units traded
    - `estimated_traded_isk`: `average × volume`

Just to get a sense of what the aggregate candidate statistics will look like, we're calculating the following features:

- Average history orders per day: total history order count divided by all calendar days in the selection window
- Average units per order: total volume divided by total history order count
- Total estimated traded ISK: the sum of daily `average × volume`
- Average price over the selection period, or period VWAP: total estimated traded ISK divided by total volume

An item can trade frequently without being expensive, or trade relatively infrequently while still moving enormous amounts of ISK. These plots should help us see how those different measures relate before deciding how to rank the candidates.

In [ ]:
activity_sql = """
with candidates as (
    select t.type_id, t.name_en as type_name
    from stg_everef_ref_types as t
    inner join manufacturing_product_ids as p on t.type_id = p.type_id
    where t.published = true and t.meta_group_id = 1
),

window_history as (
    select type_id, market_date, order_count, volume, average, estimated_traded_isk
    from int_forge_market_history
    where region_id = $region_id
        and market_date between $selection_start and $selection_end
),

activity_totals as (
    select
        c.type_id,
        c.type_name,
        count(h.market_date) as source_row_count,
        case when count(h.market_date) = 0 then 0
            else sum(h.order_count) end as total_order_count,
        case when count(h.market_date) = 0 then 0
            else sum(h.volume) end as total_volume,
        case when count(h.market_date) = 0 then 0
            else sum(h.estimated_traded_isk) end as total_estimated_traded_isk,
        case
            when sum(case when h.volume > 0 and h.average is null then 1 else 0 end) > 0 then null
            when sum(h.volume) = 0 then null
            else total_estimated_traded_isk / sum(h.volume)
        end as period_vwap
    from candidates as c
    left join window_history as h on c.type_id = h.type_id
    group by c.type_id, c.type_name
)
select *,
    total_volume / nullif(total_order_count, 0) as average_units_per_order
from activity_totals
"""

with open_study_connection() as con:
    activity = con.execute(
        activity_sql,
        {
            "region_id": REGION_ID,
            "selection_start": SELECTION_START,
            "selection_end": SELECTION_END,
        },
    ).pl()

if activity.is_empty() or activity.get_column("source_row_count").sum() == 0:
    raise ValueError("No candidate market history is available in the selection window.")

In [ ]:
relationship_activity = activity.with_columns(
    (pl.col("total_order_count").cast(pl.Float64) / SELECTION_DAYS).alias("average_history_orders_per_day")
)

In [ ]:
def prepare_relationship_fits(activity, *, x_metric, fit_metrics):
    fits = {}
    for metric in fit_metrics:
        eligible = activity.filter(
            pl.col(x_metric).cast(pl.Float64).is_finite()
            & (pl.col(x_metric) > 0)
            & pl.col(metric).cast(pl.Float64).is_finite()
            & (pl.col(metric) > 0)
        )
        xs = eligible.get_column(x_metric).to_list()
        ys = eligible.get_column(metric).to_list()
        n = len(xs)
        fit = {"n": n, "r": None, "slope": None, "intercept": None}
        fits[metric] = fit
        if n < 2 or min(xs) == max(xs) or min(ys) == max(ys):
            continue
        log_x = [math.log10(x) for x in xs]
        log_y = [math.log10(y) for y in ys]
        mean_x = math.fsum(log_x) / n
        mean_y = math.fsum(log_y) / n
        dx = [x - mean_x for x in log_x]
        dy = [y - mean_y for y in log_y]
        xx = math.fsum(x * x for x in dx)
        yy = math.fsum(y * y for y in dy)
        if xx == 0 or yy == 0:
            continue
        xy = math.fsum(x * y for x, y in zip(dx, dy))
        r = xy / math.sqrt(xx * yy)
        slope = xy / xx
        if math.isfinite(r):
            fit["r"] = max(-1.0, min(1.0, r))
        fit["slope"] = slope
        fit["intercept"] = mean_y - slope * mean_x
    return fits

In [ ]:
relationship_fits = prepare_relationship_fits(
    relationship_activity,
    x_metric="average_history_orders_per_day",
    fit_metrics={"period_vwap", "total_estimated_traded_isk"},
)

In [ ]:
plot_candidate_relationships(
    relationship_activity,
    fits=relationship_fits,
    title="Candidate activity relationships - aggregates over selection window",
    x_metric="average_history_orders_per_day",
    x_label="Average history orders per day",
    eligibility_minimum=1,
    eligibility_label="ISK-ranking eligibility minimum",
)

The three panels compare average units per order, period VWAP, and estimated traded value against average history orders per day. That daily average uses all 122 calendar days in the selection window, not just the days with recorded activity.

Both axes are logarithmic. Each panel includes candidates with positive, finite values for the measures being shown. The dashed line at 1 history order per day marks the minimum activity we'll require for the estimated-ISK ranking later on, but it doesn't filter these plots.

For the last two panels, I've added fitted lines and Pearson correlations calculated from the logged values. These summarize the relationships on the scales we're actually looking at, using all **1,088 plotted candidates** in each panel.

A few things stand out:

- **Average units per order:** Most candidates trade only a few units per order, even among the more frequently traded items. The upper band broadly consists of consumables like Antimatter Charge S and Scourge Light Missile, which are bought and sold in much larger quantities.
- **Period VWAP:** The negative relationship is fairly apparent, with a log-Pearson correlation of **−0.529**. More frequently traded items tend to be cheaper, although there's still substantial variation in price at similar order frequencies.
- **Estimated traded value:** The relationship is positive, with a log-Pearson correlation of **0.401**, but it's less tight. Items with similar order frequencies can differ by orders of magnitude in traded value, and some relatively infrequently traded items still move substantial amounts of ISK.

This gives us a reason to use both history order count and estimated traded ISK when choosing the panel, since they capture different kinds of market activity.

### 3.3 Selection rankings

We're going to rank the candidates separately by two measures over the selection window:

- **History order count:** the sum of daily market-history `order_count`
- **Estimated traded ISK:** the sum of daily VWAP multiplied by daily traded volume

The first should pick out frequently traded items, while the second should also capture expensive items that account for substantial amounts of traded ISK.

For the ISK ranking, I'll require at least one history order per day on average over the selection window. That's a minimum of 122 total history orders, rather than a requirement that the item trades every single day. This helps avoid selecting items that have very rare trades but also cost enormous amounts of ISK; removing the items with sparse activity will make the analysis less cumbersome.

After ranking, we'll select the top 50 from each set of candidates.

In [ ]:
ranking_sql = """
with isk_ranks as (
    select type_id,
        rank() over (order by total_estimated_traded_isk desc nulls last) as estimated_traded_isk_rank
    from candidate_activity
    where total_order_count >= $selection_days
)
select
    a.*,
    rank() over (order by a.total_order_count desc nulls last) as order_count_rank,
    i.estimated_traded_isk_rank
from candidate_activity as a
left join isk_ranks as i on a.type_id = i.type_id
"""

with open_study_connection() as con:
    con.register("candidate_activity", activity)
    activity = con.execute(ranking_sql, {"selection_days": SELECTION_DAYS}).pl()

top_by_order_count = activity.filter(pl.col("order_count_rank") <= TOP_N).sort(
    "total_order_count", descending=True
)
top_by_estimated_isk = activity.filter(pl.col("estimated_traded_isk_rank") <= TOP_N).sort(
    "total_estimated_traded_isk", descending=True
)

### 3.4 Activity distributions and selection cutoffs

Before looking at the individual ranked items, let's see where the top-50 cutoffs fall within the candidate distributions.

Each cumulative curve shows the percentage of candidates with totals at or below a given value. The horizontal axis is logarithmic, and the dashed line marks the lowest selected total.

The history-order chart uses all candidates with positive, finite order totals. The estimated-ISK chart uses the eligible candidates that pass our minimum order-count requirement and have positive, finite estimated traded values.

In [ ]:
order_cutoff = top_by_order_count.get_column("total_order_count").min()
isk_cutoff = top_by_estimated_isk.get_column("total_estimated_traded_isk").min()

In [ ]:
plot_activity_distribution(
    activity, metric="total_order_count", cutoff=order_cutoff,
    cutoff_text=f"{order_cutoff:,.0f} orders" if order_cutoff is not None else "", top_n=TOP_N,
    title="History order count cumulative distribution", x_label="History order count",
    y_label="Cumulative share of candidates (%)", unit="history orders", color="#3676a8",
)

The top-50 cutoff is **30,461 history orders** over the selection window, or roughly **250 per calendar day on average**. This puts our selection well into the upper tail of the candidate distribution at ~95.5% and above.

There isn't an obvious break at this point that separates active markets from everything else. The top 50 is a practical choice for keeping the study manageable, rather than a naturally distinct group of items.

In [ ]:
plot_activity_distribution(
    activity, metric="total_estimated_traded_isk", cutoff=isk_cutoff,
    cutoff_text=f"{isk_cutoff / 1e9:,.1f}B ISK" if isk_cutoff is not None else "", top_n=TOP_N,
    title="Estimated traded value cumulative distribution", x_label="Estimated traded value (ISK)",
    eligibility_column="estimated_traded_isk_rank",
    y_label="Cumulative share of eligible candidates (%)", unit="ISK", color="#288779",
)

The estimated-traded-value cutoff is approximately **120.2 billion ISK** over the selection window. We're again selecting from the upper tail at roughly 94.3%, although this curve only includes candidates that pass the minimum order-count requirement.

As with the history-order distribution, there isn't a sharp separation at the cutoff. Together, these selections give us a deliberately high-activity panel, rather than a representative sample of all Tech I markets.

### 3.5 Ranked candidates

Now let's look at which items actually made it into each ranking. The charts use separate scales, with type IDs included in the labels to distinguish items that share a name.

In [ ]:
plot_activity_ranking(
    top_by_order_count, metric="total_order_count", title=f"Top {TOP_N} by history order count",
    x_label="History order count", color="#3676a8",
)

In terms of the most ordered items, these primarily comprise fairly cheap equipment and consumable items, as one might expect. These include [ship modules](https://wiki.eveuniversity.org/Modules), [probes](https://wiki.eveuniversity.org/Probe_scanning), [drones](https://wiki.eveuniversity.org/Drones), and [missiles](https://wiki.eveuniversity.org/Missiles). Some of these also include relatively cheap beginner ships like the [Heron](https://wiki.eveuniversity.org/Heron) and [Venture](https://wiki.eveuniversity.org/Venture).

Exploration equipment is particularly prominent here. Core Probe Launcher I and Core Scanner Probe I lead the ranking, with analyzers, salvagers, and other exploration-related modules also appearing near the top.

In [ ]:
plot_activity_ranking(
    top_by_estimated_isk, metric="total_estimated_traded_isk", title=f"Top {TOP_N} by estimated traded value",
    x_label="Estimated traded value (ISK)", color="#288779",
)

Most of the total traded ISK is attributed to sales of various ships, most notably [freighters](https://wiki.eveuniversity.org/Freighters) and [battlecruisers](https://wiki.eveuniversity.org/Battlecruisers)/[battleships](https://wiki.eveuniversity.org/Battleships). These are expensive ships that often go through significant churn.

Freighters are large ships used to carry enormous quantities of cargo. Even in relatively safe "high-sec" (high-security) space, they are often targeted because they carry valuable cargo. As such, the constant destruction and replacement of these valuable ships results in substantial amounts of traded ISK.

Battlecruisers/battleships are also expensive ships and, as the name implies, often face destruction in PvE and PvP scenarios. Replacing these ships similarly creates ongoing demand.

## 4. Proposed panel

I'm combining the two top-50 lists with the seven standard mineral inputs: Tritanium, Pyerite, Mexallon, Isogen, Nocxium, Zydrine, and Megacyte. After removing duplicate items, this gives us a proposed study panel that covers both frequently traded products and products that account for substantial traded ISK, along with their primary mineral inputs. 

I'll also assign categories manually to make the resulting panel easier to describe and compare.

In [ ]:
MINERAL_IDS = [34, 35, 36, 37, 38, 39, 40]

membership = pl.concat(
    [
        pl.DataFrame({"type_id": MINERAL_IDS, "selection_reason": "curated mineral"}),
        top_by_order_count.select("type_id").with_columns(
            pl.lit("top 50 history order count").alias("selection_reason")
        ),
        top_by_estimated_isk.select("type_id").with_columns(
            pl.lit("top 50 estimated traded ISK").alias("selection_reason")
        ),
    ]
).cast({"type_id": pl.Int64, "selection_reason": pl.String})

In [ ]:
panel_sql = """
with members as (
    select
        type_id,
        bool_or(selection_reason = 'curated mineral') as is_mineral,
        bool_or(selection_reason = 'top 50 history order count') as in_order_count_top_50,
        bool_or(selection_reason = 'top 50 estimated traded ISK') as in_estimated_isk_top_50,
        string_agg(selection_reason, '; ' order by selection_reason) as selection_reason
    from membership
    group by type_id
),
window_totals as (
    select
        m.type_id,
        case when count(h.market_date) = 0 then 0
            else sum(h.order_count) end as total_order_count,
        case when count(h.market_date) = 0 then 0
            else sum(h.estimated_traded_isk) end as total_estimated_traded_isk
    from members as m
    left join int_forge_market_history as h
        on m.type_id = h.type_id
        and h.region_id = $region_id
        and h.market_date between $selection_start and $selection_end
    group by m.type_id
)
select
    m.*,
    t.name_en as type_name,
    a.order_count_rank,
    a.estimated_traded_isk_rank,
    w.total_order_count,
    w.total_estimated_traded_isk
from members as m
left join stg_everef_ref_types as t on m.type_id = t.type_id
left join activity as a on m.type_id = a.type_id
left join window_totals as w on m.type_id = w.type_id
"""

with open_study_connection() as con:
    con.register("membership", membership)
    con.register("activity", activity)
    proposed_panel = con.execute(
        panel_sql,
        {"region_id": REGION_ID, "selection_start": SELECTION_START, "selection_end": SELECTION_END},
    ).pl()

if proposed_panel.get_column("type_id").n_unique() != proposed_panel.height:
    raise ValueError("The proposed panel contains duplicate type IDs.")
if not set(MINERAL_IDS).issubset(set(proposed_panel.get_column("type_id").to_list())):
    raise ValueError("The proposed panel is missing a curated mineral.")
if not set(top_by_order_count.get_column("type_id").to_list()).issubset(
    set(proposed_panel.filter(pl.col("in_order_count_top_50")).get_column("type_id").to_list())
):
    raise ValueError("The proposed panel is missing an order-count selection.")
if not set(top_by_estimated_isk.get_column("type_id").to_list()).issubset(
    set(proposed_panel.filter(pl.col("in_estimated_isk_top_50")).get_column("type_id").to_list())
):
    raise ValueError("The proposed panel is missing an estimated-ISK selection.")

proposed_panel = proposed_panel.with_columns(
    pl.lit(REGION_ID).alias("region_id"),
    pl.lit(SELECTION_START, dtype=pl.Date).alias("selection_start"),
    pl.lit(SELECTION_END, dtype=pl.Date).alias("selection_end"),
)

### 4.1 Membership categories and ranked-selection overlap

First, let's see how much the two rankings actually overlap. The bar below separates items selected only by history order count, only by estimated traded ISK, or by both rankings.

The percentages describe the unique union of the ranked lists. The manually added minerals aren't included in this comparison.

In [ ]:
ranked_union = proposed_panel.filter(
    pl.col("in_order_count_top_50") | pl.col("in_estimated_isk_top_50")
)
ranked_union_count = ranked_union.height
if ranked_union_count == 0:
    raise ValueError("The ranked-selection union is empty.")

overlap_counts = pl.DataFrame({
    "label": ["Order-only", "Both", "ISK-only"],
    "count": [
        ranked_union.filter(pl.col("in_order_count_top_50") & ~pl.col("in_estimated_isk_top_50")).height,
        ranked_union.filter(pl.col("in_order_count_top_50") & pl.col("in_estimated_isk_top_50")).height,
        ranked_union.filter(~pl.col("in_order_count_top_50") & pl.col("in_estimated_isk_top_50")).height,
    ],
})
if overlap_counts.get_column("count").sum() != ranked_union_count:
    raise ValueError("Ranked-selection overlap does not partition the unique union.")

curated_only_additions = proposed_panel.filter(
    pl.col("is_mineral")
    & ~pl.col("in_order_count_top_50")
    & ~pl.col("in_estimated_isk_top_50")
).height

In [ ]:
plot_ranked_selection_overlap(
    overlap_counts, curated_only_count=curated_only_additions,
    title="Ranked-selection overlap", x_label="Number of items",
)

Only **5 items** appear in both rankings, with each ranking contributing **45 items** that don't appear in the other. That gives us **95 unique ranked items**, or **102 proposed items** after adding the seven minerals.

This shows a pretty substantial difference between the selections, and shows that we're analyzing a broader set of markets than we'd get from choosing either measure alone.

In [ ]:
category_ids = {
    "Minerals": [34, 35, 36, 37, 38, 39, 40],
    "Ammunition and probes": [209, 210, 211, 212, 213, 215, 216, 217, 218, 219, 220, 221, 222, 230, 22778, 30013],
    "Drones": [2183, 2203, 2454, 2464, 2486, 15508],
    "Deployables": [33474, 33475, 33700, 33702],
    "Structures": [35833],
    "Rigs": [25894, 26929, 30993, 31213, 31358, 31360, 31370, 31716, 31718, 31788, 31790],
    "Light ships": [605, 672, 49710, 54731],
    "Cruisers and battlecruisers": [621, 622, 626, 627, 16227, 16229, 16231, 24696, 24698, 24700, 24702, 47270, 54732],
    "Battleships": [638, 639, 640, 641, 642, 643, 644, 645],
    "Mining and industrial ships": [2998, 17476, 17478, 17480, 32880],
    "Freighters": [20183, 20185, 20187, 20189],
    "Mobility and concealment": [434, 439, 21857, 33985, 11370, 34126],
    "Defense and repair": [578, 2046, 3829, 4403, 40349, 41498],
    "Exploration and resource recovery": [17938, 22175, 22177, 25861, 33180, 17482, 25266],
    "Miscellaneous modules": [23527, 24283, 11563, 41492],
}

assignments = pl.DataFrame(
    [(type_id, category) for category, ids in category_ids.items() for type_id in ids],
    schema=["type_id", "study_category"],
    orient="row",
).cast({"type_id": pl.Int64, "study_category": pl.String})

if assignments.get_column("type_id").n_unique() != assignments.height:
    raise ValueError("A type ID has more than one study category.")

study_panel = proposed_panel.select("type_id", "type_name").join(
    assignments, on="type_id", how="left", validate="1:1"
)

unassigned = study_panel.filter(
    pl.col("study_category").is_null()
).select("type_id", "type_name")

if not unassigned.is_empty():
    raise ValueError("Assign a study category to each proposed item before reviewing history.")

### 4.2 Suitability: historical source-row coverage

Now that we have a proposed panel, let's check how much historical data is available for these items beyond the selection window.

The heatmap shows the percentage of calendar days in each month with a recorded market-history row for each item. A recorded day with zero activity still counts as recorded; we're checking whether the row exists, rather than how much trading occurred.

The incomplete final month is omitted so that we're comparing complete months.

In [ ]:
history_summary_sql = """
select
    p.type_id,
    p.type_name,
    min(h.market_date) as first_observed_date,
    max(h.market_date) as last_observed_date,
    count(h.market_date) as source_row_count,
    count(distinct h.market_date) as observed_day_count
from notebook_study_panel as p
left join int_forge_market_history as h
    on p.type_id = h.type_id
    and h.region_id = $region_id
    and h.market_date >= $history_start
group by p.type_id, p.type_name
"""

monthly_presence_sql = """
select
    h.type_id,
    cast(date_trunc('month', h.market_date) as date) as month,
    count(distinct h.market_date) as recorded_day_count
from int_forge_market_history as h
inner join notebook_study_panel as p on h.type_id = p.type_id
where h.region_id = $region_id and h.market_date >= $history_start
group by h.type_id, date_trunc('month', h.market_date)
"""

if study_panel.get_column("type_id").n_unique() != study_panel.height:
    raise ValueError("Study-panel membership must be unique before reviewing history.")

with open_analysis_connection() as con:
    con.register("notebook_study_panel", study_panel)
    parameters = {"region_id": REGION_ID, "history_start": SELECTION_START}
    history_summary = con.execute(history_summary_sql, parameters).pl()
    monthly_presence = con.execute(monthly_presence_sql, parameters).pl()

if history_summary.height != study_panel.height or history_summary.get_column("type_id").n_unique() != history_summary.height:
    raise ValueError("The history review must retain one row for each study-panel item.")
if history_summary.filter(pl.col("source_row_count") != pl.col("observed_day_count")).height:
    raise ValueError("Observed history contains duplicate item-date source rows.")
if monthly_presence.select(["type_id", "month"]).n_unique() != monthly_presence.height:
    raise ValueError("Monthly source-row presence must have one row per item-month.")

available_cutoff_value = history_summary.get_column("last_observed_date").drop_nulls().max()
if available_cutoff_value is None:
    raise ValueError("No observed history is available for the proposed panel.")
available_cutoff = date.fromisoformat(str(available_cutoff_value))
full_history_days = (available_cutoff - SELECTION_START).days + 1
if full_history_days <= 0:
    raise ValueError("The available history cutoff precedes the declared interval.")

if history_summary.filter(pl.col("observed_day_count") > full_history_days).height:
    raise ValueError("Observed-day coverage cannot exceed the declared interval.")

In [ ]:
months = pl.date_range(SELECTION_START, available_cutoff, interval="1mo", eager=True)
# Include the final month only when its last calendar day has been observed.
months = months.filter(months.dt.month_end() <= available_cutoff)
panel_order = study_panel.select("type_id", "type_name").with_row_index("panel_order")
presence_grid = (
    panel_order.join(pl.DataFrame({"month": months}), how="cross")
    .join(monthly_presence, on=["type_id", "month"], how="left", validate="1:1")
    .with_columns(
        pl.col("recorded_day_count").fill_null(0),
        pl.col("month").dt.month_end().dt.day().alias("expected_day_count"),
    )
    .with_columns(
        (pl.col("recorded_day_count") / pl.col("expected_day_count") * 100)
        .alias("recorded_day_pct")
    )
    .sort(["panel_order", "month"])
)
if presence_grid.filter(pl.col("recorded_day_count") > pl.col("expected_day_count")).height:
    raise ValueError("Monthly recorded days cannot exceed expected calendar days.")

In [ ]:
plot_monthly_history_availability(presence_grid)

Most of the proposed items have recorded history on nearly every day throughout most months. There are some isolated gaps, along with a shared reduction in September 2023 that coincides with EVE Online's [unplanned outage](https://forums.eveonline.com/t/resolved-unexpected-downtime-1st-september-2023/418301).

Two items stand out for having consistently fewer recorded days: the **Prototype Jump Drive Economizer** and the **Fortizar**.

### 4.3 Exclusions

#### 4.3.1 Sparse-history markets

The [Prototype Jump Drive Economizer](https://wiki.eveuniversity.org/Capital_Ship_Modules) is a Tech I module primarily used for Tech II [Jump Freighters](https://wiki.eveuniversity.org/Jump_Freighters), helping reduce their fuel costs. Since most players primarily utilize Tech I ships and components, these modules have a much more specialized audience.

The [Fortizar](https://wiki.eveuniversity.org/Upwell_structure#Citadels) is a type of player-made structure used by corporations ("guilds" in other games) and by high-risk "null-sec" players for bases. These are expensive structures that relatively few players need or want to acquire.

In [ ]:
study_panel = study_panel.filter(~pl.col("type_id").is_in([34126, 35833]))

#### 4.3.2 Civilian equipment

The 1MN Civilian Afterburner is a different kind of market. Players can repeatedly obtain it for free by stripping equipment from [corvettes](https://wiki.eveuniversity.org/Industrialist_(Producer)#Making_Mountains_of_Molehills_(3_of_10)), so its supply isn’t dependent on manufacturing. Trading probably reflects a mixture of new players buying and selling starter equipment and veterans helping them out.

Its daily prices also show unusually large swings, even during our selection window. The comparison below puts those movements alongside the other remaining items.

In [ ]:
def prepare_selection_price_changes(panel):
    sql = """
    select h.type_id, p.type_name, h.market_date, h.average as vwap
    from int_forge_market_history as h
    inner join comparison_panel as p on h.type_id = p.type_id
    where h.region_id = $region_id
        and h.market_date between $selection_start and $selection_end
    """
    with open_analysis_connection() as con:
        con.register("comparison_panel", panel)
        history = con.execute(sql, {
            "region_id": REGION_ID,
            "selection_start": SELECTION_START,
            "selection_end": SELECTION_END,
        }).pl()

    if history.select("type_id", "market_date").n_unique() != history.height:
        raise ValueError("Selection history must contain one row per item-date.")
    history = history.sort("type_id", "market_date").with_columns(
        pl.col("market_date").shift(1).over("type_id").alias("previous_date"),
        pl.col("vwap").shift(1).over("type_id").alias("previous_vwap"),
    )
    return history.filter(
        (pl.col("market_date") - pl.col("previous_date") == pl.duration(days=1))
        & pl.col("vwap").is_finite() & (pl.col("vwap") > 0)
        & pl.col("previous_vwap").is_finite() & (pl.col("previous_vwap") > 0)
    ).with_columns(
        pl.max_horizontal(
            pl.col("vwap") / pl.col("previous_vwap"),
            pl.col("previous_vwap") / pl.col("vwap"),
        ).alias("change_factor")
    )


selection_price_changes = prepare_selection_price_changes(study_panel)

In [ ]:
plot_selection_price_changes(selection_price_changes)

I’m excluding this freely available starter item to keep the panel focused on more comparable manufactured markets. This leaves **99 retained items**.

In [ ]:
study_panel = study_panel.filter(~pl.col("type_id").is_in([21857]))

## 5. Retained-panel characteristics

With the panel selected, let's get a better sense of the markets we're keeping. We're going to compare their daily prices, trading activity, and price ranges over the available history, then look at how the panel is distributed across categories.

The historical calendar runs from the selection start through the latest observed date for the retained panel. Missing history rows are treated as zero activity for volume and history order count, while prices remain missing. Values in recorded rows are preserved.

### 5.1 Daily market-history histograms

For these distributions, I'm combining the items into seven broad groups: Ships, Modules, Ammunition and probes, Deployables, Drones, Minerals, and Rigs.

Each histogram shows the distribution of daily values within each group. The bars represent percentages of that group's positive, finite item-day values, using common logarithmically spaced bins. Zero, missing, and invalid values are excluded from those percentages, with their counts available in hover.

The relative daily price range is calculated as `(high − low) / VWAP`. This describes how far apart the day's highest and lowest transaction prices were relative to its average price.

You can click the legend to hide a group, or double-click it to isolate one.

In [ ]:
retained_panel = study_panel.sort("type_id")
if retained_panel.is_empty() or retained_panel.get_column("type_id").n_unique() != retained_panel.height:
    raise ValueError("The retained study panel must contain unique items.")

retained_history_sql = """
select
    h.type_id,
    h.market_date,
    h.average as vwap,
    h.highest as daily_high,
    h.lowest as daily_low,
    h.volume,
    h.order_count
from int_forge_market_history as h
inner join notebook_retained_panel as p on h.type_id = p.type_id
where h.region_id = $region_id
    and h.market_date >= $history_start
"""

with open_analysis_connection() as con:
    con.register("notebook_retained_panel", retained_panel)
    retained_observed_history = con.execute(
        retained_history_sql,
        {"region_id": REGION_ID, "history_start": SELECTION_START},
    ).pl().with_columns(pl.lit(True).alias("_source_row_present"))

if retained_observed_history.is_empty():
    raise ValueError("No retained-panel history is available from the declared study start.")
if retained_observed_history.select(["type_id", "market_date"]).n_unique() != retained_observed_history.height:
    raise ValueError("Retained observed history must contain one source row per item-date.")

In [ ]:
retained_cutoff_value = retained_observed_history.get_column("market_date").max()
retained_history_cutoff = date.fromisoformat(str(retained_cutoff_value))
if retained_history_cutoff < SELECTION_START:
    raise ValueError("The retained-panel history cutoff precedes the declared study start.")

retained_calendar_dates = pl.date_range(
    SELECTION_START, retained_history_cutoff, interval="1d", eager=True
).cast(pl.Date)
retained_calendar = retained_panel.join(
    pl.DataFrame({"market_date": retained_calendar_dates}), how="cross"
)
retained_panel_history = retained_calendar.join(
    retained_observed_history,
    on=["type_id", "market_date"],
    how="left",
    validate="1:1",
).with_columns(
    pl.when(pl.col("_source_row_present").is_null())
    .then(pl.lit(0))
    .otherwise(pl.col("volume"))
    .alias("volume"),
    pl.when(pl.col("_source_row_present").is_null())
    .then(pl.lit(0))
    .otherwise(pl.col("order_count"))
    .alias("order_count"),
)

if retained_panel_history.height != retained_panel.height * retained_calendar_dates.len():
    raise ValueError("The retained-panel calendar has an unexpected number of item-days.")

In [ ]:
category_order = retained_panel.get_column("study_category").unique().sort().to_list()

vwap = pl.col("vwap").cast(pl.Float64, strict=False)
high = pl.col("daily_high").cast(pl.Float64, strict=False)
low = pl.col("daily_low").cast(pl.Float64, strict=False)
retained_panel_history = retained_panel_history.with_columns(
    pl.when(pl.any_horizontal(pl.col("vwap").is_null(), pl.col("daily_high").is_null(), pl.col("daily_low").is_null()))
    .then(None)
    .when(~pl.all_horizontal(vwap.is_finite(), high.is_finite(), low.is_finite()) | (vwap <= 0)
          | vwap.is_null() | high.is_null() | low.is_null())
    .then(float("nan"))
    .otherwise((high - low) / vwap)
    .alias("relative_daily_high_low_dispersion")
)

In [ ]:
plot_retained_distributions(retained_panel_history)

A few differences between these groups are pretty apparent:

- **Daily VWAP:** Ships generally occupy the higher price ranges, while ammunition and probes are concentrated toward the lower end. Several groups also have multiple concentrations, reflecting the different types of items within them.
- **Daily traded volume:** Minerals stand out for the enormous quantities traded, followed by ammunition and probes. Ships generally trade in much smaller quantities.
- **Daily history order count:** Minerals also tend to have higher daily order counts, while ships generally occupy the lower ranges. Modules, drones, rigs, and ammunition/probes overlap quite a bit.
- **Relative daily price range:** The groups overlap substantially here, although rigs appear more concentrated toward smaller relative ranges. Several groups also have long tails of days with unusually large high–low differences.

These differences give us some useful context for later anomaly analysis. What looks like unusually high volume or a large price movement for one item might be fairly ordinary for another.

### 5.2 Selection-period category composition

Finally, let's compare how much of the retained panel each category accounts for by item count, estimated traded ISK, and history order count during the original selection window.

These use the more detailed categories assigned earlier, rather than the seven broad groups used in the histograms.

In [ ]:
retained_selection = retained_panel.join(
    proposed_panel.select("type_id", "total_estimated_traded_isk", "total_order_count"),
    on="type_id",
    how="left",
    validate="1:1",
)
if retained_selection.get_column("total_estimated_traded_isk").null_count():
    raise ValueError("Each retained item needs a selection-period estimated traded ISK total.")

if retained_selection.get_column("total_order_count").null_count():
    raise ValueError("Each retained item needs a selection-period history order count.")

category_composition = retained_selection.group_by("study_category").agg(
    pl.len().alias("retained_item_count"),
    pl.col("total_order_count").sum().alias("selection_order_count"),
    pl.col("total_estimated_traded_isk").sum().alias("selection_estimated_traded_isk"),
).with_columns(
    (pl.col("retained_item_count") / pl.col("retained_item_count").sum() * 100).alias("retained_item_share_pct"),
    (
        pl.col("selection_estimated_traded_isk")
        / pl.col("selection_estimated_traded_isk").sum()
        * 100
    ).alias("selection_estimated_traded_isk_share_pct"),
    (
        pl.col("selection_order_count").cast(pl.Float64)
        / pl.col("selection_order_count").sum().cast(pl.Float64)
        * 100
    ).alias("selection_order_count_share_pct"),
).with_columns(
    pl.col("study_category").replace_strict(
        {category: index for index, category in enumerate(category_order)}
    ).alias("category_sort_order")
).sort("category_sort_order", descending=True)

if not math.isclose(category_composition.get_column("retained_item_share_pct").sum(), 100.0):
    raise ValueError("Retained-item category shares must sum to 100%.")
if not math.isclose(category_composition.get_column("selection_estimated_traded_isk_share_pct").sum(), 100.0):
    raise ValueError("Selection-period estimated traded ISK category shares must sum to 100%.")
if not math.isclose(category_composition.get_column("selection_order_count_share_pct").sum(), 100.0):
    raise ValueError("Selection-period history order count category shares must sum to 100%.")

In [ ]:
plot_retained_composition(
    category_composition, colors=CATEGORY_COLORS, title="Retained-panel composition",
    panel_titles=("Share of retained items", "Share of estimated traded value", "Share of history orders"),
    x_label="Share (%)",
)

The picture changes quite a bit depending on how we weight the categories:

- **Minerals** make up just **7% of the panel's items**, but account for approximately **48% of its estimated traded value** and **27% of its history orders**.
- **Freighters** account for around **16% of estimated traded value**, despite representing only **4% of the items** and less than **0.1% of history orders**.
- **Ammunition and probes** show the opposite pattern: roughly **19% of history orders**, but only **0.5% of estimated traded value**.

So the categories moving the most ISK aren't necessarily the ones being traded most frequently. This gives us another reason to keep both selection measures: history order count and estimated traded value capture pretty different kinds of market activity.

## 6. Production graph

So far, we've mostly looked at these item markets individually or grouped them into categories. But many of them are also connected through manufacturing.

The graph below connects our retained items to their direct production inputs and downstream manufacturing outputs. The seven minerals appear separately from the other external inputs, while products outside the study panel are grouped by their Tech/Meta classification.

In [ ]:
production_graph_links = manufacturing_links(blueprints, reference_types)

In [ ]:
plot_production_graph(
    production_graph_links, study_panel, reference_types, meta_groups, MINERAL_IDS,
    category_palette=CATEGORY_COLORS, width=1000,
)

The minerals connect to many of our retained products, giving otherwise fairly different markets a shared set of production inputs. Some products also require additional materials beyond these seven minerals.

Several retained products are themselves used to manufacture more advanced goods, including Tech II items. So these markets aren't just connected to the supply of raw materials; they're also connected to demand from downstream manufacturing.

These relationships give us something useful to examine in later analysis. For example, do products that share manufacturing inputs experience similar price movements? When a mineral's price changes substantially, do related product markets move with it, and how long does that take?

## 7. Export study seeds

With the exclusions above applied, I export the retained IDs and study categories to `transformation/seeds/forge_study_panel.csv` and production relationships to `transformation/seeds/production_links.csv`. This cell rewrites the seeds each time it runs. Note that these seeds are already committed into the repo, but if either seed changes, rebuild the downstream dbt models with `mise run transform:build` before using the frozen panel in later notebooks.

In [ ]:
from pathlib import Path

seed = study_panel.select("type_id", "study_category").sort("type_id")
repo_root = find_repo_root()

production_links = manufacturing_links(blueprints, reference_types)

seed_output_path = globals().get("SEED_OUTPUT_PATH")
seed_path = Path(seed_output_path) if seed_output_path else repo_root / "transformation" / "seeds" / "forge_study_panel.csv"
production_links_output_path = globals().get("PRODUCTION_LINKS_OUTPUT_PATH")
production_links_path = (
    Path(production_links_output_path)
    if production_links_output_path
    else repo_root / "transformation" / "seeds" / "production_links.csv"
)

seed.write_csv(seed_path, line_terminator="\n")
production_links.write_csv(production_links_path, line_terminator="\n")
seed_display_path = (
    str(seed_path.relative_to(repo_root))
    if seed_path.is_relative_to(repo_root)
    else str(seed_path)
)

final_panel = proposed_panel.join(
    study_panel.select("type_id", "study_category"), on="type_id", how="inner", validate="1:1"
)
export_summary = pl.DataFrame({
    "Frozen panel items": [study_panel.height],
    "Curated minerals": [study_panel.filter(pl.col("type_id").is_in(MINERAL_IDS)).height],
    "In both ranked lists": [final_panel.filter(
        pl.col("in_order_count_top_50") & pl.col("in_estimated_isk_top_50")
    ).height],
    "Unavailable names": [study_panel.get_column("type_name").null_count()],
    "Generated seed": [seed_display_path],
})
export_summary